# Training: Indian fillers + emotion + decisions for voice agents

This notebook builds the three parts of the stack described in the README:

- **Part 1** fine-tunes [CrisperWhisper](https://github.com/nyrahealth/crisperwhisper)
  (a Whisper variant that keeps fillers, stutters and pauses in the transcript)
  on Indian speech.
- **Part 2** trains a small emotion classification head on top of
  [emotion2vec+](https://huggingface.co/emotion2vec) utterance embeddings.
- **Part 3** wires both signals into [Laya](https://github.com/NandhaKishorM/laya),
  a multilingual decision model that turns each conversation turn into a typed
  decision (agree / disagree / inquire / ...) with calibrated probabilities.

Every turn is a decision, and "ha" (yes) vs "na" (no) vs a hesitant "hmm" sound
almost identical to a standard pipeline. This stack exists so they stop being
treated as the same.

**How to run:** open in Colab, switch to a GPU runtime (T4 is fine), then
`Runtime > Run all`. Datasets are not stored in this repo - they are streamed
at run time or supplied by you as small CSV manifests (see the README's Data
section).

## 0. Setup

Installs. Colab already ships with PyTorch; `funasr` pulls in `modelscope` for
the emotion2vec+ weights. `laya` is the decision engine used in Part 3.

In [ ]:
!pip -q install --upgrade "transformers>=4.45" "datasets>=3.0" accelerate evaluate jiwer librosa soundfile funasr modelscope scikit-learn pandas "laya>=0.5"

## Part 1 - Fine-tune CrisperWhisper on Indian speech

Stock Whisper is trained to produce clean, readable text, so it deletes fillers
by design. CrisperWhisper is trained to transcribe verbatim - every "um", "uh",
stutter and false start, with word-level timestamps - which makes it the right
backbone for an agent that needs to *hear* those sounds.

**Data options, in order of value:**

1. `data/verbatim_indic.csv` - your own clips with verbatim transcripts that
   keep the fillers (columns: `audio_path,text`). This is the only source of
   real filler supervision.
2. Fallback: stream [IndicVoices](https://huggingface.co/datasets/ai4bharat/IndicVoices)
   (gated - accept the terms and run `huggingface-cli login` first). Its
   transcripts are mostly clean/intended-style, so this adapts the model to
   Indian acoustics and code-mixing but does not teach new filler behaviour.
   To pseudo-label Indian audio with fillers at scale, use CrisperWhisper 2.0's
   `verbatimize` mode (`pip install crisperwhisper`).

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Audio, load_dataset

# --- config ---------------------------------------------------------------
ASR_MODEL_ID   = "nyrahealth/CrisperWhisper"  # verbatim Whisper variant (HF)
DATA_CSV       = "data/verbatim_indic.csv"    # columns: audio_path,text
INDIC_LANGUAGE = "hindi"                      # IndicVoices config (fallback only)
INDIC_SAMPLES  = 2000                         # streamed samples for the fallback
MAX_STEPS      = 500
OUTPUT_DIR     = "checkpoints/crisperwhisper-indic"


def load_own_csv(csv_path: str) -> Dataset:
    """Your clips with verbatim transcripts (keep fillers in the text)."""
    df = pd.read_csv(csv_path)
    assert {"audio_path", "text"} <= set(df.columns), "CSV needs audio_path and text columns"
    ds = Dataset.from_pandas(df[["audio_path", "text"]])
    return ds.cast_column("audio_path", Audio(sampling_rate=16_000)).rename_column("audio_path", "audio")


def stream_indicvoices(language: str, n: int) -> Dataset:
    """Fallback: stream spontaneous Indian speech from AI4Bharat IndicVoices.

    Gated dataset: accept the terms on the HF dataset page and authenticate
    (`huggingface-cli login` or an HF_TOKEN env var) before running this.
    """
    it = load_dataset("ai4bharat/IndicVoices", language, split="valid", streaming=True)
    sample = next(iter(it))
    print("IndicVoices columns:", list(sample.keys()))
    audio_key = "audio" if "audio" in sample else "audio_filepath"
    rows = [{"audio": r[audio_key], "text": r["text"]} for r in it.take(n)]
    ds = Dataset.from_list(rows)
    if not isinstance(ds[0]["audio"], dict):
        # only a file path was exposed - decode it via the Audio feature
        ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    return ds


raw = load_own_csv(DATA_CSV) if os.path.exists(DATA_CSV) else stream_indicvoices(INDIC_LANGUAGE, INDIC_SAMPLES)
raw = raw.train_test_split(test_size=0.05, seed=42)
raw

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(ASR_MODEL_ID)
model.config.forced_decoder_ids = None  # free language/task choice during fine-tuning
model.config.suppress_tokens = []

In [ ]:
def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


prepared = raw.map(prepare, remove_columns=raw["train"].column_names, num_proc=1)
prepared

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List

import torch


@dataclass
class Collator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        inputs = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt"
        )
        label_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt"
        )
        labels = label_batch["input_ids"].masked_fill(label_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        return {"input_features": inputs["input_features"], "labels": labels}

In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = np.where(pred.predictions[0] != -100, pred.predictions[0], processor.tokenizer.pad_token_id)
    label_ids = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_steps=MAX_STEPS,
    warmup_steps=50,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=prepared["train"],
    eval_dataset=prepared["test"],
    data_collator=Collator(processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

In [ ]:
# quick check: the fine-tuned model on a held-out clip
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model=OUTPUT_DIR,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
)
print("prediction:", asr(raw["test"][0]["audio"])["text"])
print("reference: ", raw["test"][0]["text"])

## Part 2 - Emotion head on emotion2vec+

[emotion2vec+](https://huggingface.co/emotion2vec) is a speech-emotion
foundation model trained for cross-language robustness - the "Whisper of
emotion" line of work from Alibaba DAMO. Instead of fine-tuning the whole
backbone, we extract utterance embeddings with FunASR and train a small MLP
head on our own labelled Indian clips.

**Labels.** emotion2vec+ ships with 9 classes: angry, disgusted, fearful,
happy, neutral, other, sad, surprised, unknown. Reuse them so the head stays
compatible with the pretrained model's behaviour.

**Data.** `data/emotion_indic.csv` with columns `audio_path,label`. A few
hundred clips is enough to train the head. There is no public dataset that
labels what an Indian "hmm" means - this CSV is where that knowledge enters
the system.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

EMOTION_MODEL = "iic/emotion2vec_plus_base"  # ModelScope id via FunASR; HF mirror: emotion2vec/emotion2vec_plus_base
EMOTION_CSV   = "data/emotion_indic.csv"     # columns: audio_path,label
EMOTIONS      = ["angry", "disgusted", "fearful", "happy", "neutral", "other", "sad", "surprised", "unknown"]

df = pd.read_csv(EMOTION_CSV)
assert df["label"].isin(EMOTIONS).all(), f"labels must be one of {EMOTIONS}"
df.head()

In [ ]:
from funasr import AutoModel as FunAutoModel

emo_model = FunAutoModel(model=EMOTION_MODEL, disable_update=True)


def embed(path: str) -> np.ndarray:
    """Utterance embedding: mean-pool the frame-level features."""
    out = emo_model.generate(path, granularity="utterance", extract_embedding=True)
    return np.asarray(out[0]["feats"]).mean(axis=0)


X = np.stack([embed(p) for p in df["audio_path"]])
y = df["label"].map({e: i for i, e in enumerate(EMOTIONS)}).to_numpy()
np.savez("checkpoints/emotion_embeddings.npz", X=X, y=y)
X.shape

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

device = "cuda" if torch.cuda.is_available() else "cpu"
head = nn.Sequential(
    nn.Linear(X.shape[1], 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, len(EMOTIONS)),
).to(device)

opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
Xtr_t, ytr_t = torch.tensor(Xtr, dtype=torch.float32, device=device), torch.tensor(ytr, device=device)
Xte_t, yte_t = torch.tensor(Xte, dtype=torch.float32, device=device), torch.tensor(yte, device=device)

for epoch in range(200):
    head.train()
    opt.zero_grad()
    loss = loss_fn(head(Xtr_t), ytr_t)
    loss.backward()
    opt.step()
    if (epoch + 1) % 40 == 0:
        head.eval()
        with torch.no_grad():
            acc = (head(Xte_t).argmax(-1) == yte_t).float().mean().item()
        print(f"epoch {epoch + 1:3d}  loss {loss.item():.4f}  held-out acc {acc:.3f}")

torch.save({"state_dict": head.state_dict(), "labels": EMOTIONS}, "checkpoints/emotion_head.pt")

## Part 3 - Per-turn decisions with Laya

Every user turn is a decision, and the costly confusions are between
near-identical syllables: "ha" (yes), "na" (no), a hesitant "hmm". An LLM can
make this call, but paying generation latency for a one-word answer is wasteful
and the output needs parsing.

[Laya](https://github.com/NandhaKishorM/laya) is an open-source (Apache 2.0),
multilingual, non-autoregressive decision engine
([convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya)). You
give it a state (text) and typed questions; it returns calibrated answers -
choice, score, or yes/no - in a single ~33 ms forward pass, with an abstain
flag when it would rather not commit. We feed it the verbatim transcript from
Part 1 plus the emotion label from Part 2, so "hmm" + doubtful lands on a
different branch than "hmm" + happy.

The multilingual checkpoint (`convaiinnovations/laya-multilingual`, 100+
languages) is the right default for code-mixed Indian speech.

In [ ]:
import laya

decider = laya.Agent("convaiinnovations/laya-multilingual")  # 100+ languages

TURN_QUESTIONS = {
    "intent": {
        "type": "choice",
        "instructions": "What is the caller's conversational intention in this turn?",
        "criteria": {
            "agree":     "Caller said yes, ha, ok, sure, or otherwise agreed.",
            "disagree":  "Caller said no, na, cancel, or otherwise disagreed.",
            "hesitate":  "Caller signalled doubt or thinking (hmm, uh) without committing.",
            "interrupt": "Caller said hold on, wait, or wants to stop the agent.",
            "inquire":   "Caller is asking a question or giving a complex instruction.",
        },
    },
    "confidence_check": {
        "type": "noul",
        "instructions": "Does the emotion signal contradict the words (e.g. words agree but tone is doubtful)?",
    },
}


def decide_turn(verbatim_text: str, emotion_label: str) -> dict:
    state = f"Transcript: {verbatim_text}\nDetected emotion: {emotion_label}"
    out = decider.predict(state, TURN_QUESTIONS)
    intent = out["answers"]["intent"]
    return {
        "intent": intent["choice"],
        "probabilities": intent["probabilities"],
        "abstain": intent["abstain"],   # True = route to the full LLM pipeline
        "route": intent["route"],       # "auto" or "review"
        "tone_mismatch": out["answers"]["confidence_check"],
    }


# example: two near-identical turns that must branch differently
print(decide_turn("ha ha [um] go ahead", "happy"))
print(decide_turn("na [uh] wait wait", "fearful"))

## Putting it together

Per user turn, the voice-agent runtime combines all three signals:

```json
{
  "verbatim_text": "so [um] ha, ha, do it",
  "words": [{"w": "[um]", "start": 0.42, "end": 0.61}],
  "emotion": {"label": "happy", "confidence": 0.71},
  "decision": {"intent": "agree", "route": "auto"}
}
```

The decision model answers the cheap, frequent turns (ack, refuse, interrupt)
in milliseconds and abstains on the hard ones; only abstained or complex turns
hit the LLM, and they arrive there with the fillers, timings and emotion
intact - which is what lets the agent tell an agreeing "hmm" from a doubtful
one.

## References

- CrisperWhisper: [GitHub](https://github.com/nyrahealth/crisperwhisper) · [HF model](https://huggingface.co/nyrahealth/CrisperWhisper)
- emotion2vec / emotion2vec+: [HF org](https://huggingface.co/emotion2vec) · [FunASR](https://github.com/alibaba-damo-academy/FunASR) · [paper repo](https://github.com/ddlBoJack/emotion2vec)
- Laya: [GitHub](https://github.com/NandhaKishorM/laya) · [HF model](https://huggingface.co/convaiinnovations/laya) · [docs](https://nandhakishorm.github.io/laya/)
- AI4Bharat IndicVoices: [HF dataset](https://huggingface.co/datasets/ai4bharat/IndicVoices) (CC BY 4.0, gated)